# VAZHI SFT v5.3 — Sadhguru Q&A v2 Restored (Direct Article Text)

**Key change:** Restore Sadhguru Q&A using v2 pipeline — direct article text as answers.

v5.2 dropped Sadhguru Q&A entirely due to critical quality issues in v1 (multi-agent
sonnet): 35% duplicates, 41% Q-A echo, 20% identical generic cliches. v2 bypasses
LLM generation entirely — uses cleaned article text directly as answers.

Result: 562 genuine Q&A pairs (100% unique, avg 734 words per answer) vs
v1's 1,001 pairs with only 652 unique and ~300 char answers.

| Parameter | v5.1a (previous) | v5.3 (this run) |
|-----------|-----------------|-----------------|
| Base model | CryptoYogi/vazhi-v5_0 | **CryptoYogi/vazhi-v5_1a** (has 2 epochs of Tamil SFT) |
| Dataset | v5.1 (3,888 train, no Sadhguru) | **v5.3 (3,837 train, Sadhguru Q&A v2 restored)** |
| Sadhguru Q&A | 0 (dropped) | **562 pairs (direct article text, avg 734 words)** |
| Training | 1 epoch, ~486 steps | **1 epoch, ~479 steps** |
| Lineage | vanilla -> v5.0 (1 epoch) | **vanilla -> v5.0 -> v5.1a -> v5.3** |
| Purpose | Fix mode collapse with rebalanced data | **Add long-form Tamil content (Sadhguru articles)** |


In [ ]:
# Cell 1: Dependencies
# After running this cell, RESTART the session (Runtime > Restart session / Session > Restart)
# DO NOT pin torch — Colab and Kaggle ship their own CUDA-matched versions.
# Pinning torch breaks CUDA or causes AcceleratorState errors.

!pip install -q -U \
  transformers \
  accelerate \
  peft \
  trl \
  datasets \
  huggingface_hub

print("\u2705 Dependencies installed")
print("\u26a0\ufe0f  RESTART THE SESSION NOW")
print("   Colab: Runtime \u2192 Restart session")
print("   Kaggle: Session \u2192 Restart & Clear")

In [ ]:
# Cell 2: Config + GPU Auto-Detection
#
# LoRA config: r=16, all 7 modules (4 attention + 3 MLP), LR 1e-5, 1 epoch
# v5.3 continues from v5.1a (which built on v5.0). Lineage: vanilla -> v5.0 -> v5.1a -> v5.3
# Key change: Sadhguru Q&A v2 restored (562 direct article-text pairs, avg 734 words)

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import re
import random
import glob
import gc
import shutil
import hashlib
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from huggingface_hub import login, HfApi

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainerCallback, LogitsProcessorList,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === ENVIRONMENT DETECTION ===
# Colab uses /content/, Kaggle uses /kaggle/working/
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
ENV_NAME = "Kaggle" if IS_KAGGLE else "Colab"

# === KEY CONFIG ===
BASE_MODEL = "CryptoYogi/vazhi-v5_1a"               # v5.1a model (2 epochs of Tamil SFT)
VANILLA_MODEL = "Qwen/Qwen3-0.6B"                   # Vanilla baseline for comparison
SFT_DATASET = "CryptoYogi/vazhi-tamil-sft-v5_3"     # v5.3 dataset (Sadhguru Q&A v2 restored)
OUTPUT_MODEL = "CryptoYogi/vazhi-v5_3"               # Final VAZHI model
ADAPTER_REPO = "CryptoYogi/vazhi-v5_3-lora"          # Adapter backup

# Training config
LEARNING_RATE = 1e-5       # Conservative for 0.6B (v4.2 used 5e-5 = catastrophic forgetting)
NUM_EPOCHS = 1             # Single epoch — model already has 2 epochs of Tamil SFT
MAX_LENGTH = 2048          # TRL v0.20+ renamed max_seq_length -> max_length
LORA_R = 16                # Proven in v5.0/v5.1a
LORA_ALPHA = 32            # Standard 2x ratio
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
BATCH_SIZE = 4             # Per-device (L4 has 22GB)
GRADIENT_ACCUMULATION = 2  # 4 x 1 GPU x 2 = 8 effective batch

# Qwen3 instruct <think> tokens to suppress during generation
THINK_TOKEN_IDS = [151667, 151668]

SYSTEM_PROMPT = (
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0bb5\u0bb4\u0bbf (VAZHI), "
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1 \u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bc1\u0b95\u0bcd\u0b95\u0bbe\u0ba9 "
    "AI \u0b89\u0ba4\u0bb5\u0bbf\u0baf\u0bbe\u0bb3\u0bb0\u0bcd. "
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0ba4\u0bae\u0bbf\u0bb4\u0bbf\u0bb2\u0bcd \u0baa\u0ba4\u0bbf\u0bb2\u0bb3\u0bbf\u0baa\u0bcd\u0baa\u0bc0\u0bb0\u0bcd\u0b95\u0bb3\u0bcd."
)

# GPU auto-detection
assert torch.cuda.is_available(), "GPU required! Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0).lower()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_HIGH_END_GPU = any(x in gpu_name for x in ["a100", "l4", "h100", "a10"])
USE_BF16 = IS_HIGH_END_GPU
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
n_gpus = torch.cuda.device_count()
effective_batch = BATCH_SIZE * n_gpus * GRADIENT_ACCUMULATION

print(f"\u2705 Configuration loaded")
print(f"   Environment: {ENV_NAME} (work dir: {WORK_DIR})")
print(f"   PyTorch: {torch.__version__}")
print(f"   GPU: {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB)")
print(f"   Dtype: {'bf16' if USE_BF16 else 'fp16'}")
print()
print(f"\U0001f4cb SFT v5.3 Config:")
print(f"   Base:     {BASE_MODEL} (v5.1a — 2 epochs of Tamil SFT)")
print(f"   Vanilla:  {VANILLA_MODEL} (baseline comparison)")
print(f"   Dataset:  {SFT_DATASET} (Sadhguru Q&A v2 restored, 4,264 total)")
print(f"   Output:   {OUTPUT_MODEL}")
print(f"   LR:       {LEARNING_RATE}")
print(f"   LoRA:     r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Targets:  {LORA_TARGETS}")
print(f"            (4 attention: q/k/v/o_proj + 3 MLP: gate/up/down_proj)")
print(f"   Batch:    {BATCH_SIZE} x {n_gpus} GPU x {GRADIENT_ACCUMULATION} accum = {effective_batch} effective")
print(f"   Epochs:   {NUM_EPOCHS}")
print(f"   Max len:  {MAX_LENGTH}")
print(f"   Lineage:  vanilla -> v5.0 (1 epoch) -> v5.1a (1 epoch) -> v5.3 (this run)")

In [ ]:
# Cell 3: HuggingFace Login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print(f"\u2705 Logged in via Kaggle secrets (token loaded: {hf_token is not None})")
except Exception:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
        print(f"\u2705 Logged in via Colab secrets (token loaded: {hf_token is not None})")
    except Exception:
        login()
        print("\u2705 Logged in interactively")

if hf_token is None:
    print("\u26a0\ufe0f  No token found in secrets — hub push may fail later")

In [ ]:
# Cell 4: Helper Functions — Generation, Eval, Tamil Word Validation
#
# GPT5.2 FIX: Tamil char % is broken (transliterated English scores 75-88%).
# New eval uses:
#   1. Tamil WORD validation — check if words are real Tamil (not transliterated English)
#   2. Repetition ratio — catch looping patterns
#   3. Response coherence — no code, no system tokens, reasonable length

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Check enable_thinking support
try:
    tokenizer.apply_chat_template(
        [{"role": "user", "content": "test"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    USE_THINKING_FLAG = True
except TypeError:
    USE_THINKING_FLAG = False


class SuppressThinkTokens:
    """Suppress specific token IDs by setting their logits to -inf."""
    def __init__(self, token_ids, device):
        self.suppress_ids = torch.tensor(token_ids, dtype=torch.long, device=device)

    def __call__(self, input_ids, scores):
        scores[:, self.suppress_ids] = float('-inf')
        return scores


def build_chat_prompt(user_text):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    if USE_THINKING_FLAG:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def strip_think_tags(text):
    text = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    text = re.sub(r'</?think>', '', text)
    return text.strip()


def extract_response(full_text):
    if "<|im_start|>assistant" in full_text:
        resp = full_text.split("<|im_start|>assistant")[-1]
        resp = resp.split("<|im_end|>")[0].strip()
        if resp.startswith("\n"):
            resp = resp[1:]
    else:
        resp = full_text
    return strip_think_tags(resp)


def tamil_char_pct(text):
    if not text:
        return 0.0
    return 100.0 * sum(1 for c in text if '\u0B80' <= c <= '\u0BFF') / len(text)


def compute_repeat_ratio(text, n=3):
    """Fraction of tokens in repeated n-gram chains. >0.2 is bad."""
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    seen = set()
    repeated = set()
    for i, ng in enumerate(ngrams):
        if ng in seen:
            for j in range(i, i + n):
                repeated.add(j)
        seen.add(ng)
    return len(repeated) / max(len(words), 1)


# --- GPT5.2 FIX: Tamil WORD validation ---
# Tamil char % is broken: transliterated English (\"\u0b9c\u0bc6\u0ba9\u0bcd\u0ba9\u0bc1\u0bb8\u0bcd \u0bb0\u0bc6\u0b83\u0baa\u0bcd\u0bb8\u0bcd\") scores 75-88%.
# Real Tamil words have specific structural patterns:
#   - Start with Tamil consonant/vowel (not virama/dependent vowel)
#   - Follow Tamil phonotactic rules (consonant clusters are limited)
#   - Common Tamil words use high-frequency bigrams
#
# This function checks if Tamil-script words are REAL Tamil vs transliterated.

# High-frequency Tamil bigrams (from Sangraha corpus analysis)
TAMIL_COMMON_BIGRAMS = {
    '\u0ba4\u0bae\u0bbf', '\u0bae\u0bbf\u0bb4', '\u0b95\u0bb3\u0bcd', '\u0bb5\u0bc1\u0bae\u0bcd',
    '\u0ba9\u0bcd\u0bb1', '\u0baa\u0b9f\u0bc1', '\u0b9a\u0bc6\u0baf', '\u0b95\u0bc1\u0ba4',
    '\u0bb5\u0bbe\u0b95', '\u0ba9\u0bcd\u0ba9', '\u0baa\u0bbf\u0bb0', '\u0b95\u0bbf\u0ba9',
    '\u0bae\u0bc1\u0b9f', '\u0bae\u0bbe\u0ba9', '\u0b89\u0bb0\u0bbf', '\u0ba8\u0bc0\u0b99',
    '\u0b95\u0bcd\u0b95', '\u0b9f\u0bcd\u0b9f', '\u0ba4\u0bcd\u0ba4', '\u0ba9\u0bcd\u0ba9',
    '\u0bb2\u0bcd\u0bb2', '\u0baa\u0bcd\u0baa', '\u0bae\u0bcd\u0bae', '\u0bb0\u0bcd\u0b95',
    '\u0b95\u0bcd\u0b95\u0bc1', '\u0ba8\u0bcd\u0ba4', '\u0b99\u0bcd\u0b95', '\u0bae\u0bcd\u0baa',
    '\u0ba3\u0bcd\u0b9f', '\u0bb5\u0bc7\u0ba3', '\u0b95\u0bc2\u0b9f', '\u0b87\u0bb0\u0bc1',
    '\u0b89\u0bb3\u0bcd', '\u0b92\u0bb0\u0bc1', '\u0b8e\u0ba9\u0bcd', '\u0b85\u0bb5',
}

# Tamil vowels and consonants (valid word starters)
TAMIL_VOWELS = set('\u0b85\u0b86\u0b87\u0b88\u0b89\u0b8a\u0b8e\u0b8f\u0b90\u0b92\u0b93\u0b94')
TAMIL_CONSONANTS = set('\u0b95\u0b99\u0b9a\u0b9e\u0b9f\u0ba3\u0ba4\u0ba8\u0baa\u0bae\u0baf\u0bb0\u0bb2\u0bb5\u0bb4\u0bb3\u0bb1\u0ba9\u0b9c\u0bb7\u0bb8\u0bb9')


def is_tamil_word(word):
    """Check if a word written in Tamil script is likely real Tamil.
    Returns True for real Tamil, False for transliterated English.
    """
    # Strip punctuation
    clean = re.sub(r'[^\u0B80-\u0BFF]', '', word)
    if len(clean) < 2:
        return True  # Too short to judge

    # Check 1: First char should be a valid Tamil starter
    first = clean[0]
    if first not in TAMIL_VOWELS and first not in TAMIL_CONSONANTS:
        return False

    # Check 2: Contains common Tamil bigrams
    for i in range(len(clean) - 2):
        trigram = clean[i:i+3]
        if trigram in TAMIL_COMMON_BIGRAMS:
            return True

    # Check 3: Ratio of virama (\u0bcd) — Tamil uses it frequently
    virama_count = clean.count('\u0BCD')
    if len(clean) >= 4 and virama_count == 0:
        # Long word with no virama is suspicious (transliterated words
        # often have vowel signs but no halant)
        return False

    # Default: if it starts correctly, give benefit of doubt
    return True


def tamil_word_score(text):
    """Score text based on real Tamil word percentage.
    Returns (real_tamil_pct, total_tamil_words, real_tamil_words).

    GPT5.2 fix: This catches transliterated English that passes char % checks.
    """
    words = text.split()
    tamil_words = []
    for w in words:
        tamil_chars = sum(1 for c in w if '\u0B80' <= c <= '\u0BFF')
        if tamil_chars > len(w) * 0.5:
            tamil_words.append(w)

    if not tamil_words:
        return 0.0, 0, 0

    real_count = sum(1 for w in tamil_words if is_tamil_word(w))
    return 100.0 * real_count / len(tamil_words), len(tamil_words), real_count


print("\u2705 Helper functions defined")
print(f"   Tokenizer: {len(tokenizer)} tokens")
print(f"   Tamil word validator: {len(TAMIL_COMMON_BIGRAMS)} bigrams loaded")

# Quick test of Tamil word validator
test_real = '\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bc1'  # Tamil Nadu (real Tamil)
test_fake = '\u0b9c\u0bc6\u0ba9\u0bcd\u0ba9\u0bc1\u0bb8\u0bcd'  # Genus (transliterated English)
print(f"   Test real Tamil '{test_real}': {is_tamil_word(test_real)}")
print(f"   Test transliterated '{test_fake}': {is_tamil_word(test_fake)}")

In [ ]:
# Cell 5: Pre-SFT Baseline — Vanilla AND v5.1a Model Outputs
#
# Record BOTH vanilla and v5.1a model outputs BEFORE SFT for comparison.
# Vanilla catches total regression; v5.1a shows what the base already knows.

print("\U0001f4ca Pre-SFT Baseline: Model Outputs")
print("=" * 60)

BASELINE_PROMPTS = [
    ("\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "greeting"),
    ("\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "identity"),
    ("\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "thanks"),
    ("\u0b95\u0bbe\u0bb2\u0bc8\u0baf\u0bbf\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bbe\u0baa\u0bcd\u0baa\u0bbf\u0b9f\u0bb2\u0bbe\u0bae\u0bcd?", "casual"),
    ("\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "help"),
]


def run_baseline(model_id, label):
    """Run baseline prompts on a model and return results dict."""
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=MODEL_DTYPE, device_map={"":0}, trust_remote_code=True,
    )
    mdl.eval()
    mdl.config.use_cache = True
    if hasattr(mdl, 'generation_config') and hasattr(mdl.generation_config, 'suppress_tokens'):
        mdl.generation_config.suppress_tokens = None

    suppressor = SuppressThinkTokens(THINK_TOKEN_IDS, mdl.device)
    procs = LogitsProcessorList([suppressor])

    baselines = {}
    ok_count = 0
    print(f"\n--- {label} ({model_id}) ---")
    for prompt_text, plabel in BASELINE_PROMPTS:
        full_prompt = build_chat_prompt(prompt_text)
        inputs = tokenizer(full_prompt, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            outputs = mdl.generate(
                **inputs, max_new_tokens=100, do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
                logits_processor=procs,
            )
        resp = extract_response(tokenizer.decode(outputs[0], skip_special_tokens=False))
        t_pct = tamil_char_pct(resp)
        tw_pct, tw_total, tw_real = tamil_word_score(resp)
        is_ok = len(resp) > 3 and not any(c in resp[:50] for c in ['=True', 'var ', 'function'])
        if is_ok:
            ok_count += 1
        baselines[plabel] = resp
        tag = "\u2705" if is_ok else "\u274c"
        print(f"  {tag} [{plabel}] Char:{t_pct:.0f}% Word:{tw_pct:.0f}% ({tw_real}/{tw_total})")
        print(f"     Q: {prompt_text}")
        print(f"     A: {resp[:200]}")

    print(f"  Result: {ok_count}/{len(BASELINE_PROMPTS)} coherent")

    del mdl, suppressor, procs
    gc.collect(); torch.cuda.empty_cache()
    return baselines, ok_count


# Run vanilla baseline
vanilla_baselines, vanilla_ok = run_baseline(VANILLA_MODEL, "Vanilla Qwen3-0.6B")

# Run v5.1a baseline (the model we're building on)
v51a_baselines, v51a_ok = run_baseline(BASE_MODEL, "v5.1a (pre-SFT base)")

print(f"\n{'=' * 60}")
print(f"Vanilla baseline: {vanilla_ok}/{len(BASELINE_PROMPTS)} coherent")
print(f"v5.1a baseline:   {v51a_ok}/{len(BASELINE_PROMPTS)} coherent")
if vanilla_ok < 3:
    raise RuntimeError("Vanilla model failed baseline -- do not proceed")
print("\U0001f5d1\ufe0f Baseline models freed")

In [ ]:
# Cell 6: Load & Validate Dataset v5.3
#
# v5.3 restores Sadhguru Q&A v2 (direct article text as answers):
# vazhi-packs v5 (2,958) + Sadhguru Q&A v2 (562) + conversational (200)
# + Thirukkural (169) + handcrafted (120) + behavior (60)
# + IndicAlign safety (45) + general (27)
# Total: ~4,264 samples (3,837 train + 427 eval)

print(f"\U0001f4da Loading SFT dataset from {SFT_DATASET}...")

# v5.3 has separate train/eval JSON files
try:
    sft_ds = load_dataset(SFT_DATASET)
    if "train" in sft_ds and "validation" in sft_ds:
        train_ds = sft_ds["train"]
        eval_ds = sft_ds["validation"]
    else:
        raise KeyError("No train/validation split")
except (KeyError, ValueError):
    train_ds = load_dataset("json", data_files={
        "train": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v5_3-train.json"
    })["train"]
    eval_ds = load_dataset("json", data_files={
        "eval": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v5_3-eval.json"
    })["eval"]

print(f"\u2705 Dataset loaded:")
print(f"   Train:      {len(train_ds)} samples")
print(f"   Validation: {len(eval_ds)} samples")
print(f"   Columns:    {train_ds.column_names}")

# Composition stats
bucket_dist = Counter(item.get('bucket', 'unknown') for item in train_ds)
print(f"\n\U0001f4ca Composition:")
for bucket, count in sorted(bucket_dist.items(), key=lambda x: -x[1]):
    print(f"   {bucket}: {count} ({100*count/len(train_ds):.1f}%)")

# ChatML validation
CHATML_RE = re.compile(
    r'<\|im_start\|>system\n.+?<\|im_end\|>\n'
    r'<\|im_start\|>user\n(.+?)<\|im_end\|>\n'
    r'<\|im_start\|>assistant\n(.+?)<\|im_end\|>',
    re.DOTALL
)

fail_count = 0
for i in range(len(train_ds)):
    if not CHATML_RE.search(train_ds[i]["text"]):
        fail_count += 1
        if fail_count <= 3:
            print(f"   \u274c Sample {i}: invalid ChatML")

if fail_count == 0:
    print(f"\n\u2705 All {len(train_ds)} train samples pass ChatML validation")
else:
    fail_pct = 100 * fail_count / len(train_ds)
    print(f"\n\u274c {fail_count} samples failed ChatML ({fail_pct:.1f}%)")
    if fail_pct > 1.0:
        raise RuntimeError(f"HARD ABORT: {fail_pct:.1f}% ChatML failures")

# Tamil word quality audit
print(f"\n\U0001f50d Tamil word quality audit (50 random samples):")
sample_indices = random.sample(range(len(train_ds)), min(50, len(train_ds)))
word_scores = []
for idx in sample_indices:
    m = CHATML_RE.search(train_ds[idx]["text"])
    if m:
        tw_pct, tw_total, tw_real = tamil_word_score(m.group(2))
        word_scores.append(tw_pct)

if word_scores:
    print(f"   Avg Tamil word score: {np.mean(word_scores):.1f}%")
    print(f"   Min Tamil word score: {np.min(word_scores):.1f}%")
    print(f"   Samples <50% real Tamil words: {sum(1 for s in word_scores if s < 50)}")

# Sadhguru Q&A v2 spot-check
sg_count = sum(1 for item in train_ds if item.get('bucket') == 'sadhguru_qa')
print(f"\n\U0001f50d Sadhguru Q&A v2 in train: {sg_count} samples")
if sg_count > 0:
    sg_samples = [item for item in train_ds if item.get('bucket') == 'sadhguru_qa']
    sg_word_counts = []
    for item in sg_samples[:20]:
        m = CHATML_RE.search(item["text"])
        if m:
            sg_word_counts.append(len(m.group(2).split()))
    if sg_word_counts:
        print(f"   Avg answer words (sample): {np.mean(sg_word_counts):.0f}")
        print(f"   Range: {min(sg_word_counts)}-{max(sg_word_counts)} words")

# === Convert to prompt-completion format ===
# TRL v0.20+ native completion_only_loss uses prompt/completion columns.
ASSISTANT_MARKER = "<|im_start|>assistant\n"

def split_to_prompt_completion(example):
    text = example["text"]
    idx = text.rfind(ASSISTANT_MARKER)
    if idx >= 0:
        split_point = idx + len(ASSISTANT_MARKER)
        return {"prompt": text[:split_point], "completion": text[split_point:]}
    return {"prompt": "", "completion": text}

train_ds = train_ds.map(split_to_prompt_completion, remove_columns=["text"])
eval_ds = eval_ds.map(split_to_prompt_completion, remove_columns=["text"])

# Verify conversion
empty_prompts = sum(1 for i in range(len(train_ds)) if not train_ds[i]["prompt"])
print(f"\n\u2705 Converted to prompt-completion format (TRL v0.20+ native masking)")
print(f"   Columns: {train_ds.column_names}")
print(f"   Empty prompts: {empty_prompts} (should be 0)")
if empty_prompts > 0:
    raise RuntimeError(f"ABORT: {empty_prompts} samples missing assistant marker!")

# Show a sample split
print(f"\n   Sample prompt (last 80 chars): ...{train_ds[0]['prompt'][-80:]}")
print(f"   Sample completion (first 80 chars): {train_ds[0]['completion'][:80]}...")

In [ ]:
# Cell 7: Load Model + LoRA Setup

print(f"\U0001f4e5 Loading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.padding_side = "right"

for token in ["<|im_start|>", "<|im_end|>"]:
    assert token in tokenizer.get_vocab(), f"Missing {token}!"
print(f"\u2705 Tokenizer: {len(tokenizer)} tokens, ChatML OK")

# Load model — NO device_map for training
dtype_str = 'bf16' if USE_BF16 else 'fp16'
print(f"\n\U0001f4e5 Loading {BASE_MODEL} in {dtype_str}...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=MODEL_DTYPE, trust_remote_code=True,
)
model = model.to("cuda:0")
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()

print(f"\u2705 Model: {model.num_parameters():,} params")

# LoRA — r=16, all 7 modules (4 attention + 3 MLP)
# v4.0 used same config but only 1,365 samples (overfit).
# v5.1 has 5,328 clean samples — 4x more data justifies full coverage.
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"\u2705 LoRA applied: r={LORA_R}, {len(LORA_TARGETS)} modules: {LORA_TARGETS}")

In [ ]:
# Cell 8: Dataset Preflight — Token Length + Completion Validation
#
# TRL v0.20+ handles completion-only masking natively via prompt/completion columns.
# No DataCollatorForCompletionOnlyLM needed — SFTTrainer masks prompt tokens internally.
# This cell validates the dataset is ready for training.

# Token length distribution (prompt + completion combined)
print(f"\U0001f4ca Token length distribution (train):")
token_lengths = []
prompt_lengths = []
completion_lengths = []
for idx in range(len(train_ds)):
    p_len = len(tokenizer.encode(train_ds[idx]["prompt"], add_special_tokens=False))
    c_len = len(tokenizer.encode(train_ds[idx]["completion"], add_special_tokens=False))
    prompt_lengths.append(p_len)
    completion_lengths.append(c_len)
    token_lengths.append(p_len + c_len)

token_lengths = np.array(token_lengths)
prompt_lengths = np.array(prompt_lengths)
completion_lengths = np.array(completion_lengths)

truncated = (token_lengths > MAX_LENGTH).sum()
print(f"   Total tokens — Mean: {token_lengths.mean():.0f}, Max: {token_lengths.max()}, P95: {np.percentile(token_lengths, 95):.0f}")
print(f"   Prompt tokens — Mean: {prompt_lengths.mean():.0f}, Max: {prompt_lengths.max()}")
print(f"   Completion tokens — Mean: {completion_lengths.mean():.0f}, Max: {completion_lengths.max()}")
print(f"   Truncated (>{MAX_LENGTH}): {truncated} ({100*truncated/len(train_ds):.1f}%)")

# Completion validation — ensure all completions are non-trivial
empty_completions = (completion_lengths < 3).sum()
print(f"\n\U0001f50d Completion validation:")
print(f"   Empty/trivial completions (<3 tokens): {empty_completions}")
if empty_completions > 0:
    print(f"   \u26a0\ufe0f {empty_completions} samples have near-empty completions")

# Masking ratio estimate
# With prompt-completion format, SFTTrainer masks prompt tokens (sets labels to -100)
# and only trains on completion tokens. Check the ratio is reasonable.
avg_mask_pct = 100 * prompt_lengths.mean() / token_lengths.mean()
avg_train_pct = 100 * completion_lengths.mean() / token_lengths.mean()
print(f"\n\U0001f4ca Masking estimate:")
print(f"   Masked (prompt):    ~{avg_mask_pct:.0f}% of tokens")
print(f"   Trainable (completion): ~{avg_train_pct:.0f}% of tokens")
if avg_train_pct < 10:
    print(f"   \u26a0\ufe0f Very low trainable ratio — check if completions are too short")
elif avg_train_pct > 80:
    print(f"   \u26a0\ufe0f Very high trainable ratio — check if prompts are too short")
else:
    print(f"   \u2705 Masking ratio looks healthy")

In [ ]:
# Cell 9: Preflight Mini-Training (2 steps)

print("\U0001f6e1\ufe0f Preflight: mini-training (2 steps)...")

preflight_ds = train_ds.select(range(min(200, len(train_ds))))
PREFLIGHT_DIR = f"{WORK_DIR}/preflight_sft_v5_3"

preflight_config = SFTConfig(
    output_dir=PREFLIGHT_DIR,
    num_train_epochs=1,
    max_steps=2,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    save_strategy="no",
    fp16=not USE_BF16,
    bf16=USE_BF16,
    report_to="none",
    seed=RANDOM_SEED,
    max_length=MAX_LENGTH,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

preflight_trainer = SFTTrainer(
    model=model,
    train_dataset=preflight_ds,
    args=preflight_config,
    processing_class=tokenizer,
)

preflight_result = preflight_trainer.train()
preflight_loss = preflight_result.metrics.get("train_loss", 0)

peak_vram = torch.cuda.max_memory_allocated(0) / 1e9
vram_pct = 100 * peak_vram / VRAM_GB
print(f"\u2705 Preflight complete! Loss: {preflight_loss:.4f}")
print(f"   Peak VRAM: {peak_vram:.1f} GB / {VRAM_GB:.0f} GB ({vram_pct:.0f}%)")

if vram_pct > 90:
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION = 4
    effective_batch = BATCH_SIZE * n_gpus * GRADIENT_ACCUMULATION
    print(f"   \u26a0\ufe0f VRAM > 90% -- reduced batch to {BATCH_SIZE}")

del preflight_trainer, preflight_ds
gc.collect(); torch.cuda.empty_cache()
if os.path.exists(PREFLIGHT_DIR):
    shutil.rmtree(PREFLIGHT_DIR)

In [ ]:
# Cell 10: Training Setup + Callbacks
#
# Mid-training generation check catches gibberish DURING training.
# v4.0 lesson: loss 1.43->1.03 but ALL outputs were gibberish.

steps_per_epoch = len(train_ds) // effective_batch
total_steps = steps_per_epoch * NUM_EPOCHS
log_steps = max(total_steps // 30, 5)
eval_steps = max(steps_per_epoch // 3, 10)
save_steps = max(steps_per_epoch, 20)

print(f"\U0001f4ca Training Plan:")
print(f"   Train samples:    {len(train_ds)}")
print(f"   Effective batch:  {effective_batch}")
print(f"   Steps/epoch:      ~{steps_per_epoch}")
print(f"   Total steps:      ~{total_steps}")
print(f"   Eval every:       {eval_steps} steps")
print(f"   LoRA:             r={LORA_R}, {len(LORA_TARGETS)} modules")


class LossLoggingCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if "loss" in logs:
                step = state.global_step
                loss = logs["loss"]
                lr = logs.get("learning_rate", 0)
                self.losses.append((step, loss))
                print(f"  Step {step:4d}/{total_steps} | Loss: {loss:.4f} | LR: {lr:.2e}")
            if "eval_loss" in logs:
                self.eval_losses.append((state.global_step, logs["eval_loss"]))
                print(f"  \U0001f4ca Eval Loss: {logs['eval_loss']:.4f}")


class MidTrainingGenCheck(TrainerCallback):
    """Generate actual Tamil responses mid-training to catch gibberish early."""

    SANITY_PROMPTS = [
        {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "label": "greeting"},
        {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "label": "identity"},
        {"prompt": "\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "label": "help"},
    ]

    def __init__(self, model_ref):
        self.model_ref = model_ref
        self.history = []

    def on_evaluate(self, args, state, control, **kwargs):
        step = state.global_step
        if step == 0:
            return

        print(f"\n  \U0001f50d Mid-training gen check (step {step})...")
        mdl = self.model_ref
        was_training = mdl.training

        try:
            mdl.eval()
            if hasattr(mdl, 'gradient_checkpointing_disable'):
                mdl.gradient_checkpointing_disable()
            mdl.config.use_cache = True
            if hasattr(mdl, 'generation_config'):
                gen_cfg = mdl.generation_config
                if getattr(gen_cfg, 'suppress_tokens', None) is not None:
                    gen_cfg.suppress_tokens = None

            device = next(mdl.parameters()).device
            suppressor = SuppressThinkTokens(THINK_TOKEN_IDS, device)
            procs = LogitsProcessorList([suppressor])

            garbage_count = 0
            for sp in self.SANITY_PROMPTS:
                prompt = build_chat_prompt(sp["prompt"])
                inputs = tokenizer(prompt, return_tensors="pt").to(device)
                with torch.no_grad():
                    out = mdl.generate(
                        **inputs, max_new_tokens=80, do_sample=False,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.eos_token_id,
                        logits_processor=procs,
                    )
                resp = extract_response(tokenizer.decode(out[0], skip_special_tokens=False))
                t_pct = tamil_char_pct(resp)
                tw_pct, _, _ = tamil_word_score(resp)
                rep = compute_repeat_ratio(resp)
                is_ok = len(resp) > 3 and t_pct > 15 and rep < 0.3
                if not is_ok:
                    garbage_count += 1
                tag = "\u2705" if is_ok else "\U0001f480"
                print(f"    {tag} [{sp['label']}] Char:{t_pct:.0f}% Word:{tw_pct:.0f}% Rep:{rep:.2f}")
                print(f"       {resp[:120]}")

            self.history.append({"step": step, "garbage": garbage_count})
            if garbage_count == len(self.SANITY_PROMPTS):
                print(f"  \u26a0\ufe0f  ALL GARBAGE at step {step}!")

        except Exception as e:
            print(f"  \u26a0\ufe0f  Gen check failed: {e}")
        finally:
            mdl.config.use_cache = False
            if hasattr(mdl, 'gradient_checkpointing_enable'):
                mdl.gradient_checkpointing_enable()
            if was_training:
                mdl.train()


loss_cb = LossLoggingCallback()
gen_cb = MidTrainingGenCheck(model_ref=model)

OUTPUT_DIR = f"{WORK_DIR}/vazhi-sft-v5_3"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=log_steps,
    save_steps=save_steps,
    eval_steps=eval_steps,
    eval_strategy="steps",
    save_total_limit=3,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    seed=RANDOM_SEED,
    load_best_model_at_end=False,
    dataloader_pin_memory=True,
    max_length=MAX_LENGTH,
    packing=False,
    push_to_hub=True,
    hub_model_id=ADAPTER_REPO,
    hub_strategy="every_save",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    processing_class=tokenizer,
    callbacks=[loss_cb, gen_cb],
)

print(f"\u2705 SFTTrainer ready")
print(f"   Base: {BASE_MODEL} (v5.1a — already has Tamil)")
print(f"   LR: {LEARNING_RATE}")
print(f"   LoRA: r={LORA_R}, {len(LORA_TARGETS)} modules")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   ~{total_steps} steps")

In [ ]:
# Cell 11: Run Training

print("\U0001f680 Starting SFT v5.3 training...")
print(f"   ~{total_steps} steps, {NUM_EPOCHS} epoch")
print(f"   Base: {BASE_MODEL} (v5.1a)")
print(f"   LR: {LEARNING_RATE}, Dataset: {len(train_ds)} train")
print(f"   Key change: Sadhguru Q&A v2 restored (562 long-form article pairs)")
print()

train_result = trainer.train()

print("\n\u2705 Training complete!")
metrics = train_result.metrics
for k, v in metrics.items():
    print(f"   {k}: {v}")

print("\n\U0001f4ca Final eval...")
eval_metrics = trainer.evaluate()
for k, v in eval_metrics.items():
    print(f"   {k}: {v}")

if loss_cb.losses:
    start_loss = loss_cb.losses[0][1]
    end_loss = loss_cb.losses[-1][1]
    print(f"\n\U0001f4c8 Loss: {start_loss:.4f} \u2192 {end_loss:.4f}")

train_loss = metrics.get("train_loss", end_loss)
eval_loss = eval_metrics.get("eval_loss", 0)
gap = eval_loss - train_loss
print(f"   Train: {train_loss:.4f}, Eval: {eval_loss:.4f}, Gap: {gap:.4f}")
if gap > 0.2:
    print(f"   \u26a0\ufe0f Gap > 0.2 -- possible overfitting")

In [ ]:
# Cell 11b: Continue Training — Epoch 2
#
# The initial run did 1 epoch (~479 steps). The Sadhguru Q&A v2 content (562 long-form
# articles, avg 734 words) is fundamentally new — the model needs more than 1 pass at
# LR 1e-5 to learn the long-form answer pattern.
#
# This cell continues from the trained state for 1 more epoch.
# Total after this: 2 epochs (~958 steps). Still conservative at LR 1e-5.
# (v4.2's catastrophe was LR 5e-5 × 2 epochs = 10x more aggressive gradient budget)

print("🔄 Continuing training for epoch 2...")
print(f"   Epoch 1 complete: ~{total_steps} steps done")
print(f"   Adding: 1 more epoch (~{steps_per_epoch} steps)")
print(f"   Reason: Sadhguru Q&A v2 (562 long-form articles) needs ≥2 passes at LR 1e-5")
print()

# Update config for epoch 2
EPOCH2_DIR = f"{WORK_DIR}/vazhi-sft-v5_3-epoch2"

epoch2_config = SFTConfig(
    output_dir=EPOCH2_DIR,
    num_train_epochs=1,                          # 1 more epoch
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,                  # Same LR — cosine restarts from peak
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,                            # Shorter warmup for continuation
    logging_steps=log_steps,
    save_steps=save_steps,
    eval_steps=eval_steps,
    eval_strategy="steps",
    save_total_limit=2,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    seed=RANDOM_SEED + 1,                         # Different seed for shuffle
    load_best_model_at_end=False,
    dataloader_pin_memory=True,
    max_length=MAX_LENGTH,
    packing=False,
    push_to_hub=True,
    hub_model_id=ADAPTER_REPO,
    hub_strategy="every_save",
)

# Create new callbacks for epoch 2
loss_cb2 = LossLoggingCallback()
gen_cb2 = MidTrainingGenCheck(model_ref=model)

trainer2 = SFTTrainer(
    model=model,                                  # Continues from epoch 1 state
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=epoch2_config,
    processing_class=tokenizer,
    callbacks=[loss_cb2, gen_cb2],
)

print("🚀 Starting epoch 2...")
train_result2 = trainer2.train()

print("\n✅ Epoch 2 complete!")
metrics2 = train_result2.metrics
for k, v in metrics2.items():
    print(f"   {k}: {v}")

print("\n📊 Final eval after epoch 2...")
eval_metrics2 = trainer2.evaluate()
for k, v in eval_metrics2.items():
    print(f"   {k}: {v}")

# Combined loss trajectory
if loss_cb.losses and loss_cb2.losses:
    e1_start = loss_cb.losses[0][1]
    e1_end = loss_cb.losses[-1][1]
    e2_start = loss_cb2.losses[0][1]
    e2_end = loss_cb2.losses[-1][1]
    print(f"\n📈 Combined loss trajectory:")
    print(f"   Epoch 1: {e1_start:.4f} → {e1_end:.4f}")
    print(f"   Epoch 2: {e2_start:.4f} → {e2_end:.4f}")
    print(f"   Overall: {e1_start:.4f} → {e2_end:.4f}")

train_loss = metrics2.get("train_loss", e2_end if loss_cb2.losses else 0)
eval_loss = eval_metrics2.get("eval_loss", 0)
gap = eval_loss - train_loss
print(f"   Train: {train_loss:.4f}, Eval: {eval_loss:.4f}, Gap: {gap:.4f}")
if gap > 0.2:
    print(f"   ⚠️ Gap > 0.2 -- possible overfitting")

# Update total for downstream cells
NUM_EPOCHS = 2  # Update for metadata/summary
total_steps_combined = total_steps + steps_per_epoch
print(f"\n   Total training: 2 epochs, ~{total_steps_combined} steps")

# Clean up trainer2 but keep model
del trainer2
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 12: Save LoRA Adapter

ADAPTER_PATH = f"{WORK_DIR}/vazhi-sft-v5_3-lora"

print("\U0001f4be Saving LoRA adapter...")
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

metadata = {
    "base_model": BASE_MODEL,
    "vanilla_model": VANILLA_MODEL,
    "dataset": SFT_DATASET,
    "train_samples": len(train_ds),
    "eval_samples": len(eval_ds),
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_targets": LORA_TARGETS,
    "max_length": MAX_LENGTH,
    "effective_batch": effective_batch,
    "dtype": "bf16" if USE_BF16 else "fp16",
    "train_loss": metrics.get("train_loss"),
    "eval_loss": eval_metrics.get("eval_loss"),
    "lineage": "vanilla -> v5.0 (1 epoch) -> v5.1a (1 epoch) -> v5.3 (this run)",
    "key_change": "Sadhguru Q&A v2 restored (562 direct article-text pairs)",
}
with open(f"{ADAPTER_PATH}/training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\u2705 Adapter saved")

api = HfApi()
api.create_repo(ADAPTER_REPO, exist_ok=True)
print(f"\U0001f4e4 Uploading to {ADAPTER_REPO}...")
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=ADAPTER_REPO,
    commit_message=(
        f"SFT v5.3 adapter: {BASE_MODEL} (v5.1a), r={LORA_R}, "
        f"targets={len(LORA_TARGETS)} modules, "
        f"lr={LEARNING_RATE}, {NUM_EPOCHS} epoch, {len(train_ds)} samples, "
        f"Sadhguru Q&A v2 restored"
    ),
)
print(f"\u2705 Adapter: https://huggingface.co/{ADAPTER_REPO}")

In [ ]:
# Cell 13: GPT5.2 FIX — Adapter vs Merged A/B Test
#
# Previous failures may have been caused by LoRA merge corruption.
# Test BOTH adapter inference and merged inference on the same prompts.
# If adapter works but merged doesn't, the merge is the problem.

del model, trainer
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Training model freed")

AB_PROMPTS = [
    "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd",
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?",
    "\u0b95\u0bbe\u0bb2\u0bc8\u0baf\u0bbf\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bbe\u0baa\u0bcd\u0baa\u0bbf\u0b9f\u0bb2\u0bbe\u0bae\u0bcd?",
    "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd",
    "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf",
]

# --- Test A: Adapter inference (no merge) ---
print("\n" + "=" * 60)
print("\U0001f1e6 TEST A: Adapter Inference (no merge)")
print("=" * 60)

base_a = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0}, trust_remote_code=True,
)
adapter_model = PeftModel.from_pretrained(base_a, ADAPTER_PATH)
adapter_model.eval()
adapter_model.config.use_cache = True
if hasattr(adapter_model, 'generation_config'):
    adapter_model.generation_config.suppress_tokens = None

suppressor = SuppressThinkTokens(THINK_TOKEN_IDS, adapter_model.device)
procs = LogitsProcessorList([suppressor])

adapter_results = []
for prompt_text in AB_PROMPTS:
    inputs = tokenizer(build_chat_prompt(prompt_text), return_tensors="pt").to(adapter_model.device)
    with torch.no_grad():
        out = adapter_model.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=procs,
        )
    resp = extract_response(tokenizer.decode(out[0], skip_special_tokens=False))
    t_pct = tamil_char_pct(resp)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    adapter_results.append({"resp": resp, "char": t_pct, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Char:{t_pct:.0f}% Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

del adapter_model, base_a
gc.collect(); torch.cuda.empty_cache()

# --- Test B: Merged model ---
print("\n" + "=" * 60)
print("\U0001f1e7 TEST B: Merged Model (fp16)")
print("=" * 60)

base_b = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0}, trust_remote_code=True,
)
peft_b = PeftModel.from_pretrained(base_b, ADAPTER_PATH)
peft_b.gradient_checkpointing_disable()
peft_b.config.use_cache = True
peft_b.eval()

print("\U0001f500 Merging LoRA in fp16...")
merged_model = peft_b.merge_and_unload()
if hasattr(merged_model, 'generation_config'):
    merged_model.generation_config.suppress_tokens = None

suppressor = SuppressThinkTokens(THINK_TOKEN_IDS, merged_model.device)
procs = LogitsProcessorList([suppressor])

merged_results = []
for prompt_text in AB_PROMPTS:
    inputs = tokenizer(build_chat_prompt(prompt_text), return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        out = merged_model.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=procs,
        )
    resp = extract_response(tokenizer.decode(out[0], skip_special_tokens=False))
    t_pct = tamil_char_pct(resp)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    merged_results.append({"resp": resp, "char": t_pct, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Char:{t_pct:.0f}% Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

del peft_b, base_b
gc.collect(); torch.cuda.empty_cache()

# --- A/B Comparison ---
print("\n" + "=" * 60)
print("\U0001f4ca A/B COMPARISON: Adapter vs Merged")
print(f"   LoRA: r={LORA_R}, {len(LORA_TARGETS)} modules — more params to merge")
print("=" * 60)
for i, prompt in enumerate(AB_PROMPTS):
    a = adapter_results[i]
    b = merged_results[i]
    match = "\u2705" if abs(a["char"] - b["char"]) < 10 else "\u26a0\ufe0f DIVERGED"
    print(f"  {match} Q: {prompt[:40]}")
    print(f"     Adapter: Char:{a['char']:.0f}% Word:{a['word']:.0f}% Rep:{a['rep']:.2f}")
    print(f"     Merged:  Char:{b['char']:.0f}% Word:{b['word']:.0f}% Rep:{b['rep']:.2f}")

avg_a_char = np.mean([r["char"] for r in adapter_results])
avg_b_char = np.mean([r["char"] for r in merged_results])
print(f"\n  Adapter avg Tamil char: {avg_a_char:.0f}%")
print(f"  Merged avg Tamil char:  {avg_b_char:.0f}%")
if abs(avg_a_char - avg_b_char) > 15:
    print(f"  \u26a0\ufe0f MERGE CORRUPTION DETECTED! Use adapter inference instead.")
    MERGE_OK = False
else:
    print(f"  \u2705 Merge looks healthy (difference < 15%)")
    MERGE_OK = True

In [ ]:
# Cell 14: Full Eval — 16 Conversational Prompts with Tamil Word Validation
#
# Tamil WORD validation (not just char %) catches transliterated English gibberish.

merged_model.eval()
merged_model.config.use_cache = True
if hasattr(merged_model, 'generation_config'):
    merged_model.generation_config.suppress_tokens = None

think_suppressor = SuppressThinkTokens(THINK_TOKEN_IDS, merged_model.device)
logits_procs = LogitsProcessorList([think_suppressor])

test_prompts = [
    {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "check": "greeting", "cat": "greeting"},
    {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "check": "identity", "cat": "greeting"},
    {"prompt": "\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "general", "cat": "help"},
    {"prompt": "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "check": "general", "cat": "help"},
    {"prompt": "\u0ba8\u0bbe\u0ba9\u0bcd \u0b92\u0bb0\u0bc1 \u0baa\u0bbf\u0bb0\u0b9a\u0bcd\u0b9a\u0ba9\u0bc8 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b95\u0bb5\u0bb2\u0bc8\u0baa\u0bcd\u0baa\u0b9f\u0bc1\u0b95\u0bbf\u0bb1\u0bc7\u0ba9\u0bcd. \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf\u0bb2\u0bbe\u0bae\u0bcd?", "check": "general", "cat": "help"},
    {"prompt": "\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1. \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0bb5\u0ba4\u0bc1?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0bb5\u0bc0\u0b9f\u0bcd\u0b9f\u0bbf\u0bb2\u0bcd \u0ba4\u0bc0 \u0bb5\u0bbf\u0baa\u0ba4\u0bcd\u0ba4\u0bc1 \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0ba8\u0bbe\u0bb3\u0bc8 \u0baa\u0b99\u0bcd\u0b95\u0bc1 \u0b9a\u0ba8\u0bcd\u0ba4\u0bc8 \u0b8f\u0bb1\u0bc1\u0bae\u0bbe?", "check": "refusal", "cat": "refusal"},
    {"prompt": "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "govt"},
    {"prompt": "\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "check": "domain", "cat": "govt"},
    {"prompt": "\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "health"},
    {"prompt": "\u0b95\u0bbe\u0baf\u0bcd\u0b9a\u0bcd\u0b9a\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "domain", "cat": "health"},
    {"prompt": "\u0b95\u0bb2\u0bcd\u0bb5\u0bbf \u0b95\u0b9f\u0ba9\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "edu"},
    {"prompt": "\u0b95\u0bbe\u0bb2\u0bc8\u0baf\u0bbf\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bbe\u0baa\u0bcd\u0baa\u0bbf\u0b9f\u0bb2\u0bbe\u0bae\u0bcd?", "check": "general", "cat": "general"},
    {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd \u0bae\u0bca\u0bb4\u0bbf \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "general", "cat": "general"},
    {"prompt": "\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "general", "cat": "culture"},
]

print(f"\n{'=' * 60}")
print(f"\U0001f9ea SFT v5.3 EVAL: {len(test_prompts)} prompts")
print(f"   Using: Tamil WORD validation")
print(f"   Base: {BASE_MODEL} (v5.1a)")
print(f"   Dataset: {SFT_DATASET} (Sadhguru Q&A v2 restored)")
print(f"{'=' * 60}")

results = []
for tp in test_prompts:
    inputs = tokenizer(build_chat_prompt(tp["prompt"]), return_tensors="pt").to(merged_model.device)

    gen_kwargs = dict(
        max_new_tokens=150,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        logits_processor=logits_procs,
        no_repeat_ngram_size=4,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.2,
    )
    if tp["check"] in ("greeting", "identity"):
        gen_kwargs["do_sample"] = False
        for k in ["temperature", "top_p", "repetition_penalty"]:
            gen_kwargs.pop(k, None)

    with torch.no_grad():
        outputs = merged_model.generate(**inputs, **gen_kwargs)

    resp = extract_response(tokenizer.decode(outputs[0], skip_special_tokens=False))
    t_char = tamil_char_pct(resp)
    tw_pct, tw_total, tw_real = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)

    # Quality checks
    issues = []
    if len(resp.strip()) < 5:
        issues.append("empty")
    if rep > 0.2:
        issues.append("repetitive")
    if any(c in resp[:100] for c in ['=True', 'var ', 'function', '<br', 'import ']):
        issues.append("code_garbage")
    if "<think>" in resp:
        issues.append("think_leak")
    if tw_total > 0 and tw_pct < 40:
        issues.append(f"low_tamil_words({tw_pct:.0f}%)")
    if t_char < 15 and len(resp) > 10:
        issues.append("low_tamil_chars")

    passed = len(issues) == 0
    results.append({"cat": tp["cat"], "prompt": tp["prompt"], "resp": resp,
                    "passed": passed, "issues": issues,
                    "char": t_char, "word": tw_pct, "rep": rep})

    tag = "\u2705" if passed else "\u274c"
    print(f"\n{tag} [{tp['cat']}] Char:{t_char:.0f}% Word:{tw_pct:.0f}%({tw_real}/{tw_total}) Rep:{rep:.2f}")
    if issues:
        print(f"   Issues: {', '.join(issues)}")
    print(f"  Q: {tp['prompt']}")
    print(f"  A: {resp[:250]}")

In [ ]:
# Cell 15: Eval Summary + Comparison with Vanilla AND v5.1a Baselines

pass_count = sum(1 for r in results if r["passed"])
avg_char = np.mean([r["char"] for r in results])
avg_word = np.mean([r["word"] for r in results])
avg_rep = np.mean([r["rep"] for r in results])

print(f"\n{'=' * 60}")
print(f"\U0001f4ca SFT v5.3 EVAL SUMMARY")
print(f"{'=' * 60}")
print(f"   Passed:          {pass_count}/{len(results)} ({100*pass_count/len(results):.0f}%)")
print(f"   Avg Tamil char:  {avg_char:.0f}%")
print(f"   Avg Tamil word:  {avg_word:.0f}% (catches transliterated English)")
print(f"   Avg repeat:      {avg_rep:.2f}")
print(f"   Merge OK:        {'\u2705' if MERGE_OK else '\u274c'}")

# Category breakdown
cat_stats = defaultdict(lambda: {"pass": 0, "total": 0})
for r in results:
    cat_stats[r["cat"]]["total"] += 1
    if r["passed"]:
        cat_stats[r["cat"]]["pass"] += 1

print("\n   Category breakdown:")
for cat, stats in sorted(cat_stats.items()):
    pct = 100 * stats["pass"] / stats["total"]
    print(f"     {cat:10s}: {stats['pass']}/{stats['total']} ({pct:.0f}%)")

# Compare with BOTH baselines
print(f"\n   Vanilla baseline comparison:")
for label, vanilla_resp in vanilla_baselines.items():
    v_char = tamil_char_pct(vanilla_resp)
    v_word, _, _ = tamil_word_score(vanilla_resp)
    print(f"     {label}: Vanilla Char:{v_char:.0f}% Word:{v_word:.0f}%")

print(f"\n   v5.1a baseline comparison (pre-SFT base):")
for label, v51a_resp in v51a_baselines.items():
    v_char = tamil_char_pct(v51a_resp)
    v_word, _, _ = tamil_word_score(v51a_resp)
    print(f"     {label}: v5.1a Char:{v_char:.0f}% Word:{v_word:.0f}%")

# Issue summary
all_issues = []
for r in results:
    all_issues.extend(r["issues"])
if all_issues:
    print("\n   Issue frequency:")
    for issue, count in Counter(all_issues).most_common():
        print(f"     {issue}: {count}")

# Pass/fail criteria
c_overall = pass_count / len(results) >= 0.60
c_word = avg_word > 40
c_rep = avg_rep < 0.15

print(f"\n\U0001f4cb Pass Criteria:")
print(f"   {'\u2705' if c_overall else '\u274c'} Overall >= 60%: {100*pass_count/len(results):.0f}%")
print(f"   {'\u2705' if c_word else '\u274c'} Avg Tamil WORD > 40%: {avg_word:.0f}%")
print(f"   {'\u2705' if c_rep else '\u274c'} Avg repeat < 0.15: {avg_rep:.2f}")
print(f"   {'\u2705' if MERGE_OK else '\u274c'} Merge integrity: {'OK' if MERGE_OK else 'CORRUPTED'}")

EVAL_PASSED = c_overall and c_word and c_rep and MERGE_OK

print(f"\n   Training lineage:")
print(f"   v4.0: 12/12 'passed' (metric-only) but gibberish \u274c")
print(f"   v4.1: 16/16 'passed' but still gibberish \u274c")
print(f"   v4.2: 16/16 'passed' but transliterated English \u274c")
print(f"   v5.0: vanilla -> v5.0 (1 epoch, 666 steps)")
print(f"   v5.1a: v5.0 -> v5.1a (1 epoch, ~486 steps)")
print(f"   v5.3: v5.1a -> v5.3 (1 epoch, ~{total_steps} steps) {'\u2705' if EVAL_PASSED else '\u274c'}")
print(f"   v5.3 eval: {pass_count}/{len(results)}, Tamil word: {avg_word:.0f}%")

if EVAL_PASSED:
    print(f"\n\U0001f389 SFT v5.3 PASSED! Proceed to upload and GGUF.")
else:
    print(f"\n\u274c SFT v5.3 did not pass.")
    print(f"   Check: loss curve, mid-training gen history, A/B test results")

In [ ]:
# Cell 16: Upload Merged Model

if not EVAL_PASSED:
    print("\u274c Eval did not pass. Skipping upload.")
    print(f"   Adapter available at: {ADAPTER_REPO}")
else:
    api = HfApi()
    api.create_repo(OUTPUT_MODEL, exist_ok=True)

    print(f"\U0001f4e4 Pushing merged fp16 model to {OUTPUT_MODEL}...")
    merged_model.push_to_hub(
        OUTPUT_MODEL,
        private=False,
        commit_message=(
            f"SFT v5.3: VAZHI Tamil assistant, "
            f"base={BASE_MODEL} (v5.1a, 2 epochs Tamil SFT), "
            f"LoRA r={LORA_R} x {len(LORA_TARGETS)} modules, "
            f"lr={LEARNING_RATE}, {NUM_EPOCHS} epoch, "
            f"{len(train_ds)} samples (v5.3: Sadhguru Q&A v2 restored), "
            f"eval: {pass_count}/{len(results)}, "
            f"tamil_word: {avg_word:.0f}%"
        ),
    )
    tokenizer.push_to_hub(OUTPUT_MODEL)

    print(f"\n\u2705 Model:   https://huggingface.co/{OUTPUT_MODEL}")
    print(f"\u2705 Adapter: https://huggingface.co/{ADAPTER_REPO}")
    print(f"\n\U0001f449 Next: Convert to GGUF (Q4_K_M) for mobile deployment")

print(f"\n{'=' * 60}")
print(f"\U0001f4cb SFT v5.3 Summary")
print(f"{'=' * 60}")
print(f"  Lineage: vanilla -> v5.0 (1 epoch) -> v5.1a (1 epoch) -> v5.3 (this run)")
print(f"  Base:    {BASE_MODEL} (v5.1a — 2 epochs of Tamil SFT)")
print(f"  Dataset: {SFT_DATASET} (Sadhguru Q&A v2 restored)")
print(f"  Key:     562 genuine article-text Q&A pairs (avg 734 words)")
print(f"  LR:      {LEARNING_RATE}")
print(f"  Epochs:  {NUM_EPOCHS}")
print(f"  LoRA:    r={LORA_R}, {len(LORA_TARGETS)} modules: {LORA_TARGETS}")
print(f"  Eval:    {pass_count}/{len(results)}, Tamil word: {avg_word:.0f}%")
print(f"  Merge:   {'OK' if MERGE_OK else 'CORRUPTED'}")

In [ ]:
# Cell 17: Resume from Checkpoint (use ONLY if training was interrupted)
#
# Uncomment and run only if Colab disconnected mid-training.

# checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=os.path.getmtime)
# if checkpoints:
#     latest = checkpoints[-1]
#     print(f"Resuming from {latest}")
#     train_result = trainer.train(resume_from_checkpoint=latest)

print("Cell 17: Resume cell (commented out). Uncomment only if training interrupted.")